模型的预测：模型是一个 token 一个 token 生成的,流式输出能让我们看到这个过程

In [1]:
from dotenv import load_dotenv
from zai import ZhipuAiClient
import zai

# 提前加载环境变量
load_dotenv()

client = ZhipuAiClient()

In [2]:
def stream_chat(prompt: str):
    # chunk 计数器
    chunk_count: int = 0

    try:
        response = client.chat.completions.create(
            model="glm-4.5-air",
            messages=[
                {'role': 'user', 'content': prompt},
            ],
            stream=True,
        )

        for chunk in response:
            chunk_count = chunk_count + 1
            if chunk.usage: # 流结束 统计token用量
                prompt_tokens = chunk.usage.prompt_tokens
                completion_tokens = chunk.usage.completion_tokens
                total_tokens = chunk.usage.total_tokens
                print(f"\n用量：token:{total_tokens}(提示词:{prompt_tokens} 回复:{completion_tokens}), chunk:{chunk_count}")
            if chunk.choices and chunk.choices[0].delta.content:
                yield chunk.choices[0].delta.content
                # 不能用return，return会直接结束掉函数
                # yield 生成器，交给外部去遍历 
    except zai.core.APIStatusError as err:
        return f"API 状态错误: {err}"
    except zai.core.APITimeoutError as err:
        return f"请求超时: {err}"
    except Exception as err:
        return f"其他错误: {err}" 

In [4]:
res = stream_chat("为什么会发生地震？")
for r in res:
        print(r, end='', flush=True)

地震的发生是地球内部能量释放的一种自然现象，其根本原因在于**地球岩石圈的构造运动**。以下是导致地震发生的主要机制和原因：

### 🌍 1.  板块构造运动（最主要原因）
*   **理论基础：** 地球最外层的岩石圈（包括地壳和上地幔顶部）并非一个整体，而是被分割成若干巨大的板块（如太平洋板块、欧亚板块、印度板块、美洲板块、非洲板块、南极洲板块等）。
*   **板块运动：** 这些板块漂浮在下方更具塑性的软流圈上，以每年几厘米的速度缓慢移动。这种运动的驱动力源于地幔对流（地球内部热能驱动的物质缓慢流动）。
*   **板块相互作用：** 板块边界是板块相互作用最剧烈、最集中的地方。板块之间的相互作用主要有三种基本类型：
    *   **汇聚边界（碰撞/俯冲）：**
        *   **碰撞：** 两个大陆板块相向移动，相互挤压、碰撞、褶皱、抬升，形成巨大的山脉（如喜马拉雅山脉就是印度板块与欧亚板块碰撞形成的）。这种碰撞过程中，岩石被强烈挤压、弯曲、断裂，积累巨大应力，最终突然释放，引发**逆冲型地震**。
        *   **俯冲：** 一个大洋板块（密度较大）向另一个板块（通常是大陆板块或另一个大洋板块）之下俯冲插入。俯冲板块弯曲、断裂，与上覆板块发生摩擦和剪切，同时俯冲板块内部也可能发生断裂。这些过程都会释放巨大能量，引发**浅源、中源、深源地震**（深度可达700公里）。环太平洋地震带就是典型的俯冲带，全球约90%的地震发生在这里。
    *   **分离边界（张裂）：**
        *   两个板块相互背离移动，地幔物质上涌，形成新的洋壳。新形成的岩石比较脆弱，容易在拉张力作用下发生断裂。这种断裂称为**正断层**，释放能量引发**正断层型地震**。例如，大西洋中脊、东非大裂谷等。
    *   **转换边界（走滑）：**
        *   两个板块沿水平方向相互错动、剪切，但既不显著增生也不显著消减。岩石在巨大的剪切应力作用下发生断裂和错动。这种断裂称为**走滑断层**，释放能量引发**走滑型地震**。例如，美国圣安德烈斯断层就是著名的转换边界，发生大量地震。
*   **板块内部：** 虽然板块内部相对稳定，但并非完全平静。板块内部同样存在古老的断层带或应力集中区域，当应力积累到一定程度超过岩石强度时，也会发生